[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agent-sdk/02-agentes-compuestos.ipynb)

# Agentes Compuestos: Orquestador y Subagentes

Implementamos un sistema multi-agente donde un orquestador coordina subagentes especializados.

In [ ]:
import anthropic
from typing import Callable
import time

client = anthropic.Anthropic()
print("Sistema multi-agente listo")

## 1. Factory de subagentes

In [ ]:
def crear_subagente(
    rol: str,
    herramientas: list,
    ejecutor_herramientas: Callable,
    modelo: str = "claude-haiku-4-5-20251001"
) -> Callable:
    """Factory que crea un subagente con rol y herramientas específicas."""

    def subagente(tarea: str, contexto: str = "") -> str:
        system = f"""Eres un especialista en {rol}.
{'Contexto adicional: ' + contexto if contexto else ''}
Completa la tarea asignada usando las herramientas disponibles."""

        mensajes = [{"role": "user", "content": tarea}]

        for _ in range(15):
            resp = client.messages.create(
                model=modelo,
                max_tokens=1500,
                system=system,
                tools=herramientas if herramientas else anthropic.NOT_GIVEN,
                messages=mensajes
            )
            mensajes.append({"role": "assistant", "content": resp.content})

            if resp.stop_reason == "end_turn":
                return next((b.text for b in resp.content if hasattr(b, "text")), "")

            if resp.stop_reason == "tool_use":
                resultados = []
                for bloque in resp.content:
                    if bloque.type == "tool_use":
                        resultado = ejecutor_herramientas(bloque.name, bloque.input)
                        resultados.append({"type": "tool_result",
                                           "tool_use_id": bloque.id,
                                           "content": str(resultado)})
                mensajes.append({"role": "user", "content": resultados})

        return "Tarea completada (máx iteraciones)"

    return subagente

print("Factory de subagentes definida")

## 2. Definir subagentes especializados

In [ ]:
# Herramientas del investigador
HERRAMIENTAS_INVESTIGADOR = [
    {"name": "buscar_noticias",
     "description": "Busca noticias recientes sobre un tema",
     "input_schema": {"type": "object",
                      "properties": {"tema": {"type": "string"},
                                     "dias": {"type": "integer", "default": 7}},
                      "required": ["tema"]}},
    {"name": "buscar_datos_mercado",
     "description": "Obtiene datos de mercado y estadísticas",
     "input_schema": {"type": "object",
                      "properties": {"sector": {"type": "string"},
                                     "metrica": {"type": "string"}},
                      "required": ["sector"]}}
]

# Herramientas del analista
HERRAMIENTAS_ANALISTA = [
    {"name": "calcular_tendencia",
     "description": "Calcula tendencia de una serie de datos",
     "input_schema": {"type": "object",
                      "properties": {"datos": {"type": "array", "items": {"type": "number"}},
                                     "periodo": {"type": "string"}},
                      "required": ["datos"]}},
    {"name": "comparar_competidores",
     "description": "Compara métricas entre empresas competidoras",
     "input_schema": {"type": "object",
                      "properties": {"empresas": {"type": "array", "items": {"type": "string"}},
                                     "metrica": {"type": "string"}},
                      "required": ["empresas", "metrica"]}}
]

def ejecutor_investigador(nombre: str, params: dict) -> str:
    if nombre == "buscar_noticias":
        return (f"Noticias sobre '{params['tema']}' (últimos {params.get('dias', 7)} días): "
                f"[Crecimiento del sector 15% anual, nueva regulación en 2025, "
                f"inversión récord de 2.3B€, 3 startups en Serie B]")
    elif nombre == "buscar_datos_mercado":
        return (f"Datos de mercado {params['sector']}: TAM 45B€, CAGR 12%, "
                f"3 jugadores principales con 65% cuota, "
                f"margen medio del sector 35%")
    return "Datos no disponibles"

def ejecutor_analista(nombre: str, params: dict) -> str:
    if nombre == "calcular_tendencia":
        datos = params["datos"]
        if len(datos) >= 2:
            crecimiento = (datos[-1] - datos[0]) / datos[0] * 100
            return (f"Tendencia: {crecimiento:+.1f}% total, "
                    f"{crecimiento/len(datos):+.1f}% por período. "
                    f"Punto máximo: {max(datos)}, mínimo: {min(datos)}")
        return "Insuficientes datos para calcular tendencia"
    elif nombre == "comparar_competidores":
        empresas = params["empresas"]
        return (f"Comparativa {params['metrica']} entre {empresas}: "
                f"{empresas[0]} lidera con 34% cuota, {empresas[1] if len(empresas) > 1 else 'Empresa B'} "
                f"en segundo lugar con 28%")
    return "Análisis no disponible"


# Crear los subagentes
subagente_investigador = crear_subagente(
    "investigación de mercado y recopilación de datos",
    HERRAMIENTAS_INVESTIGADOR,
    ejecutor_investigador
)

subagente_analista = crear_subagente(
    "análisis de datos e interpretación de tendencias",
    HERRAMIENTAS_ANALISTA,
    ejecutor_analista
)

# Subagente redactor sin herramientas
subagente_redactor = crear_subagente(
    "redacción ejecutiva de informes de negocio",
    [],
    lambda n, p: ""
)

print("Subagentes creados: investigador, analista, redactor")

## 3. Orquestador de análisis de mercado

In [ ]:
def orquestador_analisis_mercado(sector: str) -> dict:
    """Orquestador que coordina el pipeline de análisis de mercado."""
    print(f"\n🎯 Analizando mercado: {sector}")
    inicio = time.time()

    # Fase 1: Investigación
    print("\n📡 [Subagente Investigador] Recopilando datos...")
    info_mercado = subagente_investigador(
        f"Recopila información sobre el mercado de {sector} en Europa: "
        f"tamaño de mercado, tendencias, noticias recientes y principales actores."
    )
    print(f"   → {info_mercado[:120]}...")

    # Fase 2: Análisis
    print("\n📊 [Subagente Analista] Procesando datos...")
    analisis = subagente_analista(
        f"Analiza esta información de mercado y extrae insights clave con datos: {info_mercado[:500]}",
        contexto=f"Sector: {sector}"
    )
    print(f"   → {analisis[:120]}...")

    # Fase 3: Redacción del informe
    print("\n✍️ [Subagente Redactor] Redactando informe ejecutivo...")
    informe = subagente_redactor(
        f"""Redacta un informe ejecutivo para inversores con estos datos:

Investigación de mercado: {info_mercado}

Análisis: {analisis}

El informe debe tener: Resumen ejecutivo (3 líneas), Oportunidad de mercado, 
Riesgos principales, Recomendación de inversión."""
    )

    duracion = time.time() - inicio
    print(f"\n⏱️ Pipeline completado en {duracion:.1f}s")

    return {
        "sector": sector,
        "investigacion": info_mercado,
        "analisis": analisis,
        "informe_ejecutivo": informe
    }


resultado = orquestador_analisis_mercado("inteligencia artificial B2B en Europa")
print("\n" + "="*60)
print("INFORME EJECUTIVO")
print("="*60)
print(resultado["informe_ejecutivo"])

## 4. Contexto compartido entre agentes

In [ ]:
class ContextoCompartido:
    """Almacén de contexto que los agentes pueden leer y escribir."""

    def __init__(self):
        self._datos = {}
        self._historial = []

    def guardar(self, clave: str, valor):
        self._datos[clave] = valor
        self._historial.append(f"ESCRITURA: {clave}")
        print(f"  💾 Contexto guardado: {clave}")

    def leer(self, clave: str, default=None):
        return self._datos.get(clave, default)

    def resumen(self) -> str:
        return "\n".join(
            f"- {k}: {str(v)[:150]}"
            for k, v in self._datos.items()
        )

    def claves(self) -> list:
        return list(self._datos.keys())


# Pipeline con contexto compartido
contexto = ContextoCompartido()

print("Pipeline: Análisis de competidores SaaS RRHH")
print("-" * 50)

# Paso 1: investigar competidores
print("\n[Paso 1] Investigando competidores...")
datos_competidores = subagente_investigador(
    "Busca los principales competidores en el mercado SaaS de RRHH en España: "
    "nombres, cuota de mercado estimada, precio y propuesta de valor principal"
)
contexto.guardar("competidores_raw", datos_competidores)

# Paso 2: analizar con contexto del paso anterior
print("\n[Paso 2] Analizando con contexto previo...")
analisis_comp = subagente_analista(
    "Analiza fortalezas y debilidades de estos competidores y calcula tendencias",
    contexto=f"Datos previos:\n{contexto.resumen()}"
)
contexto.guardar("analisis_competidores", analisis_comp)

print(f"\n📋 Claves en contexto: {contexto.claves()}")
print("\n=== ANÁLISIS FINAL ===")
print(analisis_comp[:500])

## 5. Delegación dinámica

In [ ]:
SUBAGENTES = {
    "investigador": subagente_investigador,
    "analista": subagente_analista,
    "redactor": subagente_redactor
}

HERRAMIENTA_DELEGACION = {
    "name": "delegar_tarea",
    "description": "Delega una subtarea a un agente especializado",
    "input_schema": {
        "type": "object",
        "properties": {
            "agente": {
                "type": "string",
                "enum": ["investigador", "analista", "redactor"],
                "description": "Agente especializado al que delegar"
            },
            "tarea": {"type": "string", "description": "Descripción detallada de la tarea"}
        },
        "required": ["agente", "tarea"]
    }
}

def ejecutar_delegacion(nombre: str, params: dict) -> str:
    if nombre == "delegar_tarea":
        agente_nombre = params["agente"]
        print(f"    → Delegando a [{agente_nombre}]: {params['tarea'][:60]}...")
        if agente_nombre in SUBAGENTES:
            return SUBAGENTES[agente_nombre](params["tarea"])
        return f"Agente '{agente_nombre}' no disponible"
    return "Herramienta desconocida"


# Orquestador que delega dinámicamente
def orquestador_dinamico(peticion: str) -> str:
    mensajes = [{"role": "user", "content": peticion}]
    print(f"\n🎯 Petición: {peticion}")

    for i in range(10):
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1500,
            system="""Eres un orquestador que coordina agentes especializados.
Para tareas de investigación de mercado usa el agente 'investigador'.
Para análisis numérico usa el agente 'analista'.
Para redactar informes finales usa el agente 'redactor'.
Delega inteligentemente y sintetiza los resultados.""",
            tools=[HERRAMIENTA_DELEGACION],
            messages=mensajes
        )
        mensajes.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            return next((b.text for b in resp.content if hasattr(b, "text")), "")

        if resp.stop_reason == "tool_use":
            resultados = []
            for bloque in resp.content:
                if bloque.type == "tool_use":
                    resultado = ejecutar_delegacion(bloque.name, bloque.input)
                    resultados.append({"type": "tool_result",
                                       "tool_use_id": bloque.id,
                                       "content": resultado[:800]})
            mensajes.append({"role": "user", "content": resultados})

    return "Completado"

# Demo de delegación dinámica
resp_final = orquestador_dinamico(
    "Necesito un análisis completo del mercado de CRM en España: "
    "investiga el mercado, analiza los datos y redacta un resumen ejecutivo"
)
print("\n" + resp_final)